In [2]:
import pandas as pd

# 1. Read the CSV (adjust path if needed)
path = "/kaggle/input/datasets/mobeenfatimah/supply-chain-operation-and-delivery-performance-100k/supply_chain_logistics_delay_dataset.csv"
df = pd.read_csv(path)

# 2. Basic shape and columns
print("Shape:", df.shape)
print("\nColumns (%d):" % len(df.columns))
for i, c in enumerate(df.columns, 1):
    print(f"{i:>3}. {c}")

# 3. Dtypes + missing values summary
summary = pd.DataFrame({
    "dtype": df.dtypes,
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique()
})
print("\nColumn summary:")
print(summary)

# 4. Peek at first rows
print("\nHead:")
print(df.head(3).T)  # transposed so 106 columns are readable

# 5. Key target fields
for col in ["delivery_delayed", "delay_hours", "delay_days"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].describe(include="all"))
        if df[col].nunique() < 20:
            print(df[col].value_counts(dropna=False))

Shape: (100000, 106)

Columns (106):
  1. shipment_id
  2. order_id
  3. tracking_number
  4. shipment_date
  5. expected_delivery_date
  6. actual_delivery_date
  7. product_id
  8. product_category
  9. product_subcategory
 10. order_priority
 11. order_quantity
 12. unit_price
 13. order_value
 14. package_weight_kg
 15. package_volume_m3
 16. origin_country
 17. origin_state
 18. origin_city
 19. origin_warehouse
 20. origin_latitude
 21. origin_longitude
 22. destination_country
 23. destination_state
 24. destination_city
 25. destination_region
 26. destination_latitude
 27. destination_longitude
 28. distance_km
 29. transport_mode
 30. vehicle_type
 31. carrier_id
 32. vehicle_age
 33. driver_experience_years
 34. fuel_type
 35. fuel_efficiency
 36. carrier_rating
 37. carrier_reliability_score
 38. carrier_experience_years
 39. historical_delay_rate
 40. on_time_delivery_rate
 41. supplier_id
 42. supplier_country
 43. supplier_rating
 44. supplier_reliability_score
 45. supp

In [3]:
import pandas as pd

path = "/kaggle/input/datasets/mobeenfatimah/supply-chain-operation-and-delivery-performance-100k/supply_chain_logistics_delay_dataset.csv"
df = pd.read_csv(path)

# --- 1. delay_days actual counts (the thing you asked about) ---
print("=== delay_days value counts ===")
print(df["delay_days"].value_counts().sort_index())
print("\nTotal delayed shipments (delay_days > 0):", (df["delay_days"] > 0).sum())
print("Total on-time (delay_days == 0):", (df["delay_days"] == 0).sum())

# --- 2. delay_reason distribution ---
print("\n=== delay_reason value counts ===")
print(df["delay_reason"].value_counts(dropna=False))

# --- 3. Only the columns in our schema ---
schema_cols = [
    # fact keys + measures + flags
    "shipment_id", "shipment_date", "order_quantity", "order_value",
    "total_logistics_cost", "distance_km", "delay_hours", "delay_days",
    "delivery_attempts", "delivery_delayed", "delivery_successful",
    "customs_required", "delay_reason",
    # dim_carrier
    "carrier_id", "carrier_rating", "carrier_reliability_score",
    "transport_mode", "vehicle_type", "vehicle_age",
    # dim_supplier
    "supplier_id", "supplier_country", "supplier_rating", "supplier_lead_time_days",
    # dim_route
    "route_id", "route_type", "route_quality", "road_condition",
    "traffic_congestion_level", "weather_condition",
    # dim_product
    "product_category", "product_subcategory", "order_priority",
]

present = [c for c in schema_cols if c in df.columns]
missing_from_schema = [c for c in schema_cols if c not in df.columns]
print("\nSchema columns present:", len(present))
print("Schema columns NOT found:", missing_from_schema)

# --- 4. Cardinality + top value for each schema column ---
rows = []
for c in present:
    s = df[c]
    rows.append({
        "column": c,
        "dtype": str(s.dtype),
        "n_unique": s.nunique(),
        "top_value": s.value_counts(dropna=False).index[0],
        "top_freq_pct": round(s.value_counts(dropna=False, normalize=True).iloc[0] * 100, 2),
        "missing": s.isna().sum(),
    })
summary = pd.DataFrame(rows).sort_values("n_unique")
print("\n=== SCHEMA COLUMN SUMMARY (sorted by cardinality) ===")
print(summary.to_string(index=False))

# --- 5. Critical dimensions: is there enough variation to slice on? ---
for col in ["carrier_id", "supplier_id", "route_id", "transport_mode",
            "vehicle_type", "route_type", "weather_condition",
            "traffic_congestion_level", "product_category", "order_priority"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].value_counts().head(10))

=== delay_days value counts ===
delay_days
0    74702
1     3668
2     3632
3     3612
4     3589
5     3562
6     3637
7     3598
Name: count, dtype: int64

Total delayed shipments (delay_days > 0): 25298
Total on-time (delay_days == 0): 74702

=== delay_reason value counts ===
delay_reason
No Delay                74702
Traffic                  7067
Weather                  4128
Warehouse Congestion     3100
Supplier Delay           2994
Carrier Issue            2022
Customs                  2016
Documentation Error      1019
Vehicle Breakdown        1013
Inventory Shortage        988
Route Disruption          463
Labor Shortage            249
Port Congestion           239
Name: count, dtype: int64

Schema columns present: 32
Schema columns NOT found: []

=== SCHEMA COLUMN SUMMARY (sorted by cardinality) ===
                   column   dtype  n_unique                  top_value  top_freq_pct  missing
      delivery_successful    bool         2                       True         98.02 

In [4]:
import pandas as pd

path = "/kaggle/input/datasets/mobeenfatimah/supply-chain-operation-and-delivery-performance-100k/supply_chain_logistics_delay_dataset.csv"
df = pd.read_csv(path)

cost_cols = ["fuel_cost", "transportation_cost", "warehouse_cost",
             "handling_cost", "customs_cost", "insurance_cost"]

# 1. Do the components sum to the total?
df["component_sum"] = df[cost_cols].sum(axis=1)
df["diff"] = df["total_logistics_cost"] - df["component_sum"]
df["abs_diff"] = df["diff"].abs()

print("=== DIFF (total - sum of components) ===")
print(df["diff"].describe())
print("\nRows with |diff| <= 1 :", (df["abs_diff"] <= 1).sum(), "/", len(df))
print("Rows with |diff| > 1  :", (df["abs_diff"] > 1).sum())
print("Max abs diff:", df["abs_diff"].max())

# 2. How much of the total do components explain?
df["coverage_pct"] = (df["component_sum"] / df["total_logistics_cost"] * 100).round(2)
print("\n=== COVERAGE % (component_sum / total) ===")
print(df["coverage_pct"].describe())

# 3. Peek at the worst mismatches
print("\n=== TOP 5 MISMATCH ROWS ===")
print(df.nlargest(5, "abs_diff")[["shipment_id", "total_logistics_cost", "component_sum", "diff"]])

# 4. Component cost totals (for your cost-breakdown story)
print("\n=== COMPONENT TOTALS ===")
print(df[cost_cols].sum().sort_values(ascending=False).round(2))
print("\nShare of component sum (%):")
print((df[cost_cols].sum() / df[cost_cols].sum().sum() * 100).sort_values(ascending=False).round(2))

=== DIFF (total - sum of components) ===
count    1.000000e+05
mean     1.036184e-15
std      1.703661e-12
min     -1.455192e-11
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.455192e-11
Name: diff, dtype: float64

Rows with |diff| <= 1 : 100000 / 100000
Rows with |diff| > 1  : 0
Max abs diff: 1.4551915228366852e-11

=== COVERAGE % (component_sum / total) ===
count    100000.0
mean        100.0
std           0.0
min         100.0
25%         100.0
50%         100.0
75%         100.0
max         100.0
Name: coverage_pct, dtype: float64

=== TOP 5 MISMATCH ROWS ===
       shipment_id  total_logistics_cost  component_sum          diff
366   SHP-0000016F              71419.19       71419.19 -1.455192e-11
1376  SHP-00000561              66649.85       66649.85  1.455192e-11
1858  SHP-00000743              83721.20       83721.20 -1.455192e-11
2668  SHP-00000A6D              75399.16       75399.16  1.455192e-11
2703  SHP-00000A90              72694.21       726

In [5]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
import os

# ---------- 1. LOAD ----------
path = "/kaggle/input/datasets/mobeenfatimah/supply-chain-operation-and-delivery-performance-100k/supply_chain_logistics_delay_dataset.csv"
df = pd.read_csv(path)

# Parse dates early
df["shipment_date"] = pd.to_datetime(df["shipment_date"], errors="coerce")

# ---------- 2. CLEAN / DERIVE ----------
# Yes/No -> 1/0
df["delivery_delayed_flag"] = (df["delivery_delayed"].str.strip().str.lower() == "yes").astype(int)
df["delivery_successful_flag"] = df["delivery_successful"].astype(int)
df["customs_required_flag"] = (df["customs_required"].str.strip().str.lower() == "yes").astype(int)

# Vehicle age bucket
bins = [-1, 2, 5, 10, 100]
labels = ["0-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"]
df["vehicle_age_bucket"] = pd.cut(df["vehicle_age"], bins=bins, labels=labels).astype(str)

# ---------- 3. DIM_DATE ----------
min_d, max_d = df["shipment_date"].min(), df["shipment_date"].max()
dates = pd.date_range(min_d, max_d, freq="D")
dim_date = pd.DataFrame({"date": dates})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%b")
dim_date["week"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["day_of_week"] = dim_date["date"].dt.day_name()
dim_date["is_weekend"] = dim_date["date"].dt.weekday.isin([5, 6]).astype(int)

# ---------- 4. DIMENSIONS ----------
def make_dim(frame, cols, key_name):
    d = frame[cols].drop_duplicates().reset_index(drop=True)
    d[key_name] = range(1, len(d) + 1)
    return d

dim_carrier = make_dim(
    df,
    ["carrier_id", "carrier_rating", "carrier_reliability_score",
     "transport_mode", "vehicle_type", "vehicle_age_bucket"],
    "carrier_key",
)

dim_supplier = make_dim(
    df,
    ["supplier_id", "supplier_country", "supplier_rating", "supplier_lead_time_days"],
    "supplier_key",
)

dim_route = make_dim(
    df,
    ["route_id", "route_type", "route_quality", "road_condition",
     "traffic_congestion_level", "weather_condition"],
    "route_key",
)

dim_product = make_dim(
    df,
    ["product_category", "product_subcategory", "order_priority"],
    "product_key",
)

# ---------- 5. MAP KEYS INTO BASE ROW ----------
base = df.merge(dim_carrier, on=["carrier_id", "carrier_rating", "carrier_reliability_score",
                                 "transport_mode", "vehicle_type", "vehicle_age_bucket"], how="left")
base = base.merge(dim_supplier, on=["supplier_id", "supplier_country", "supplier_rating",
                                    "supplier_lead_time_days"], how="left")
base = base.merge(dim_route, on=["route_id", "route_type", "route_quality", "road_condition",
                                 "traffic_congestion_level", "weather_condition"], how="left")
base = base.merge(dim_product, on=["product_category", "product_subcategory", "order_priority"], how="left")
base["date_key"] = base["shipment_date"].dt.strftime("%Y%m%d").astype("Int64")

# ---------- 6. FACT_SHIPMENT ----------
fact_shipment = base[[
    "shipment_id", "date_key", "carrier_key", "supplier_key", "route_key", "product_key",
    "order_quantity", "order_value", "total_logistics_cost", "distance_km",
    "delay_hours", "delay_days", "delivery_attempts",
    "delivery_delayed_flag", "delivery_successful_flag", "customs_required_flag",
    "delay_reason",
]].rename(columns={
    "delivery_delayed_flag": "delivery_delayed",
    "delivery_successful_flag": "delivery_successful",
    "customs_required_flag": "customs_required",
})

# ---------- 7. FACT_COST (unpivot) ----------
cost_cols = ["fuel_cost", "transportation_cost", "warehouse_cost",
             "handling_cost", "customs_cost", "insurance_cost"]

fact_cost = base[["shipment_id", "date_key", "carrier_key", "supplier_key",
                  "route_key", "product_key"] + cost_cols].melt(
    id_vars=["shipment_id", "date_key", "carrier_key", "supplier_key",
             "route_key", "product_key"],
    value_vars=cost_cols,
    var_name="cost_type",
    value_name="cost_amount",
)

# tidy cost_type labels
cost_type_map = {
    "fuel_cost": "Fuel",
    "transportation_cost": "Transportation",
    "warehouse_cost": "Warehouse",
    "handling_cost": "Handling",
    "customs_cost": "Customs",
    "insurance_cost": "Insurance",
}
fact_cost["cost_type"] = fact_cost["cost_type"].map(cost_type_map)

# ---------- 8. EXPORT ----------
out_dir = "/kaggle/working/pbi_supply_chain"
os.makedirs(out_dir, exist_ok=True)

tables = {
    "Dim_Date": dim_date.drop(columns=["date"]),
    "Dim_Carrier": dim_carrier,
    "Dim_Supplier": dim_supplier,
    "Dim_Route": dim_route,
    "Dim_Product": dim_product,
    "Fact_Shipment": fact_shipment,
    "Fact_Cost": fact_cost,
}

for name, t in tables.items():
    t.to_csv(f"{out_dir}/{name}.csv", index=False)
    print(f"{name}: {t.shape[0]:,} rows × {t.shape[1]} cols")

# ---------- 9. ZIP ----------
zip_path = "/kaggle/working/pbi_supply_chain_tables.zip"
with ZipFile(zip_path, "w") as z:
    for name in tables:
        z.write(f"{out_dir}/{name}.csv", arcname=f"{name}.csv")

print("\nZIP created:", zip_path)
print("Size (MB):", round(os.path.getsize(zip_path) / 1e6, 2))

Dim_Date: 1,641 rows × 8 cols
Dim_Carrier: 18,034 rows × 7 cols
Dim_Supplier: 550 rows × 5 cols
Dim_Route: 1,440 rows × 7 cols
Dim_Product: 160 rows × 4 cols
Fact_Shipment: 100,000 rows × 17 cols
Fact_Cost: 600,000 rows × 8 cols

ZIP created: /kaggle/working/pbi_supply_chain_tables.zip
Size (MB): 42.49


In [6]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
import os

# ---------- 1. LOAD ----------
path = "/kaggle/input/datasets/mobeenfatimah/supply-chain-operation-and-delivery-performance-100k/supply_chain_logistics_delay_dataset.csv"
df = pd.read_csv(path)
df["shipment_date"] = pd.to_datetime(df["shipment_date"], errors="coerce")

# ---------- 2. CLEAN / DERIVE ----------
df["delivery_delayed"] = (df["delivery_delayed"].str.strip().str.lower() == "yes").astype(int)
df["delivery_successful"] = df["delivery_successful"].astype(int)
df["customs_required"] = (df["customs_required"].str.strip().str.lower() == "yes").astype(int)

bins = [-1, 2, 5, 10, 100]
labels = ["0-2 yrs", "3-5 yrs", "6-10 yrs", "10+ yrs"]
df["vehicle_age_bucket"] = pd.cut(df["vehicle_age"], bins=bins, labels=labels).astype(str)

# ---------- 3. DIM_DATE ----------
min_d, max_d = df["shipment_date"].min(), df["shipment_date"].max()
dates = pd.date_range(min_d, max_d, freq="D")
dim_date = pd.DataFrame({"date": dates})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%b")
dim_date["week"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["day_of_week"] = dim_date["date"].dt.day_name()
dim_date["is_weekend"] = dim_date["date"].dt.weekday.isin([5, 6]).astype(int)

# ---------- 4. DIMENSIONS (correct grain) ----------
# Carrier: one row per carrier_id
dim_carrier = df[["carrier_id", "carrier_rating", "carrier_reliability_score"]] \
    .drop_duplicates(subset=["carrier_id"]).reset_index(drop=True)
dim_carrier["carrier_key"] = range(1, len(dim_carrier) + 1)

# Supplier: one row per supplier_id
dim_supplier = df[["supplier_id", "supplier_country", "supplier_rating", "supplier_lead_time_days"]] \
    .drop_duplicates(subset=["supplier_id"]).reset_index(drop=True)
dim_supplier["supplier_key"] = range(1, len(dim_supplier) + 1)

# Route: one row per route_id
dim_route = df[["route_id", "route_type", "route_quality", "road_condition"]] \
    .drop_duplicates(subset=["route_id"]).reset_index(drop=True)
dim_route["route_key"] = range(1, len(dim_route) + 1)

# Product: one row per category/subcategory/priority combo
dim_product = df[["product_category", "product_subcategory", "order_priority"]] \
    .drop_duplicates().reset_index(drop=True)
dim_product["product_key"] = range(1, len(dim_product) + 1)

# ---------- 5. MAP ALL KEYS INTO BASE ----------
base = df.merge(dim_carrier, on=["carrier_id", "carrier_rating", "carrier_reliability_score"], how="left")
base = base.merge(dim_supplier, on=["supplier_id", "supplier_country", "supplier_rating",
                                    "supplier_lead_time_days"], how="left")
base = base.merge(dim_route, on=["route_id", "route_type", "route_quality", "road_condition"], how="left")
base = base.merge(dim_product, on=["product_category", "product_subcategory", "order_priority"], how="left")
base["date_key"] = base["shipment_date"].dt.strftime("%Y%m%d").astype("Int64")

# Sanity check: no null keys
for k in ["date_key", "carrier_key", "supplier_key", "route_key", "product_key"]:
    print(k, "nulls:", base[k].isna().sum())

# ---------- 6. FACT_SHIPMENT ----------
fact_shipment = base[[
    "shipment_id", "date_key", "carrier_key", "supplier_key", "route_key", "product_key",
    "order_quantity", "order_value", "total_logistics_cost", "distance_km",
    "delay_hours", "delay_days", "delivery_attempts",
    "delivery_delayed", "delivery_successful", "customs_required", "delay_reason",
    # varying attributes moved out of dimensions:
    "transport_mode", "vehicle_type", "vehicle_age_bucket",
    "weather_condition", "traffic_congestion_level",
]]

# ---------- 7. FACT_COST ----------
cost_cols = ["fuel_cost", "transportation_cost", "warehouse_cost",
             "handling_cost", "customs_cost", "insurance_cost"]

fact_cost = base[["shipment_id", "date_key", "carrier_key", "supplier_key",
                  "route_key", "product_key"] + cost_cols].melt(
    id_vars=["shipment_id", "date_key", "carrier_key", "supplier_key",
             "route_key", "product_key"],
    value_vars=cost_cols,
    var_name="cost_type",
    value_name="cost_amount",
)

fact_cost["cost_type"] = fact_cost["cost_type"].map({
    "fuel_cost": "Fuel",
    "transportation_cost": "Transportation",
    "warehouse_cost": "Warehouse",
    "handling_cost": "Handling",
    "customs_cost": "Customs",
    "insurance_cost": "Insurance",
})

# ---------- 8. EXPORT + ZIP ----------
out_dir = "/kaggle/working/pbi_supply_chain"
os.makedirs(out_dir, exist_ok=True)

tables = {
    "Dim_Date": dim_date.drop(columns=["date"]),
    "Dim_Carrier": dim_carrier,
    "Dim_Supplier": dim_supplier,
    "Dim_Route": dim_route,
    "Dim_Product": dim_product,
    "Fact_Shipment": fact_shipment,
    "Fact_Cost": fact_cost,
}

for name, t in tables.items():
    t.to_csv(f"{out_dir}/{name}.csv", index=False)
    print(f"{name}: {t.shape[0]:,} rows × {t.shape[1]} cols")

zip_path = "/kaggle/working/pbi_supply_chain_tables.zip"
with ZipFile(zip_path, "w") as z:
    for name in tables:
        z.write(f"{out_dir}/{name}.csv", arcname=f"{name}.csv")

print("\nZIP:", zip_path, "| Size (MB):", round(os.path.getsize(zip_path) / 1e6, 2))

date_key nulls: 0
carrier_key nulls: 0
supplier_key nulls: 0
route_key nulls: 0
product_key nulls: 0
Dim_Date: 1,641 rows × 8 cols
Dim_Carrier: 120 rows × 4 cols
Dim_Supplier: 550 rows × 5 cols
Dim_Route: 60 rows × 5 cols
Dim_Product: 160 rows × 4 cols
Fact_Shipment: 100,000 rows × 22 cols
Fact_Cost: 600,000 rows × 8 cols

ZIP: /kaggle/working/pbi_supply_chain_tables.zip | Size (MB): 43.3


In [7]:
print(df["vehicle_age"].describe())
print(df["vehicle_age"].value_counts().sort_index())
print("Bucketed counts:")
print(df["vehicle_age_bucket"].value_counts().sort_index())

count    100000.000000
mean          6.007550
std           3.167269
min           1.000000
25%           3.000000
50%           6.000000
75%           9.000000
max          11.000000
Name: vehicle_age, dtype: float64
vehicle_age
1     9151
2     9116
3     8903
4     9103
5     9087
6     8976
7     9257
8     9001
9     9152
10    9009
11    9245
Name: count, dtype: int64
Bucketed counts:
vehicle_age_bucket
0-2 yrs     18267
10+ yrs      9245
3-5 yrs     27093
6-10 yrs    45395
Name: count, dtype: int64


In [11]:
# Carrier stability
print("=== CARRIER ===")
for c in ["carrier_rating", "carrier_reliability_score"]:
    print(c, df.groupby("carrier_id")[c].nunique().max())

# Route stability
print("\n=== ROUTE ===")
for c in ["route_type", "route_quality", "road_condition"]:
    print(c, df.groupby("route_id")[c].nunique().max())

# Supplier stability
print("\n=== SUPPLIER ===")
for c in ["supplier_country", "supplier_rating", "supplier_lead_time_days"]:
    print(c, df.groupby("supplier_id")[c].nunique().max())

# Product / order_priority
print("\n=== PRODUCT / PRIORITY ===")
print("order_priority per product_id:", df.groupby("product_id")["order_priority"].nunique().max())
print("product_subcategory per product_id:", df.groupby("product_id")["product_subcategory"].nunique().max())
print("product_category per product_id:", df.groupby("product_id")["product_category"].nunique().max())

=== CARRIER ===
carrier_rating 1
carrier_reliability_score 1

=== ROUTE ===
route_type 1
route_quality 1
road_condition 1

=== SUPPLIER ===
supplier_country 1
supplier_rating 1
supplier_lead_time_days 1

=== PRODUCT / PRIORITY ===
order_priority per product_id: 1
product_subcategory per product_id: 1
product_category per product_id: 1


In [12]:
import pandas as pd
import numpy as np
from zipfile import ZipFile
import os

# ---------- 1. LOAD ----------
path = "/kaggle/input/datasets/mobeenfatimah/supply-chain-operation-and-delivery-performance-100k/supply_chain_logistics_delay_dataset.csv"
df = pd.read_csv(path)
df["shipment_date"] = pd.to_datetime(df["shipment_date"], errors="coerce")

# ---------- 2. CLEAN / DERIVE ----------
df["delivery_delayed"] = (df["delivery_delayed"].str.strip().str.lower() == "yes").astype(int)
df["delivery_successful"] = df["delivery_successful"].astype(int)
df["customs_required"] = (df["customs_required"].str.strip().str.lower() == "yes").astype(int)

# FIXED: vehicle age buckets (1-11 range, balanced)
bins = [0, 3, 6, 9, 11]
labels = ["1-3 yrs", "4-6 yrs", "7-9 yrs", "10-11 yrs"]
df["vehicle_age_bucket"] = pd.cut(df["vehicle_age"], bins=bins, labels=labels,
                                  include_lowest=True).astype(str)

# ---------- 3. DIM_DATE ----------
min_d, max_d = df["shipment_date"].min(), df["shipment_date"].max()
dates = pd.date_range(min_d, max_d, freq="D")
dim_date = pd.DataFrame({"date": dates})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["date"].dt.year
dim_date["quarter"] = dim_date["date"].dt.quarter
dim_date["month"] = dim_date["date"].dt.month
dim_date["month_name"] = dim_date["date"].dt.strftime("%b")
dim_date["week"] = dim_date["date"].dt.isocalendar().week.astype(int)
dim_date["day_of_week"] = dim_date["date"].dt.day_name()
dim_date["is_weekend"] = dim_date["date"].dt.weekday.isin([5, 6]).astype(int)

# ---------- 4. DIMENSIONS ----------
dim_carrier = df[["carrier_id", "carrier_rating", "carrier_reliability_score"]] \
    .drop_duplicates(subset=["carrier_id"]).reset_index(drop=True)
dim_carrier["carrier_key"] = range(1, len(dim_carrier) + 1)

dim_supplier = df[["supplier_id", "supplier_country", "supplier_rating", "supplier_lead_time_days"]] \
    .drop_duplicates(subset=["supplier_id"]).reset_index(drop=True)
dim_supplier["supplier_key"] = range(1, len(dim_supplier) + 1)

dim_route = df[["route_id", "route_type", "route_quality", "road_condition"]] \
    .drop_duplicates(subset=["route_id"]).reset_index(drop=True)
dim_route["route_key"] = range(1, len(dim_route) + 1)

dim_product = df[["product_category", "product_subcategory", "order_priority"]] \
    .drop_duplicates().reset_index(drop=True)
dim_product["product_key"] = range(1, len(dim_product) + 1)

# ---------- 5. MAP KEYS ----------
base = df.merge(dim_carrier, on=["carrier_id", "carrier_rating", "carrier_reliability_score"], how="left")
base = base.merge(dim_supplier, on=["supplier_id", "supplier_country", "supplier_rating",
                                    "supplier_lead_time_days"], how="left")
base = base.merge(dim_route, on=["route_id", "route_type", "route_quality", "road_condition"], how="left")
base = base.merge(dim_product, on=["product_category", "product_subcategory", "order_priority"], how="left")
base["date_key"] = base["shipment_date"].dt.strftime("%Y%m%d").astype("Int64")

for k in ["date_key", "carrier_key", "supplier_key", "route_key", "product_key"]:
    print(k, "nulls:", base[k].isna().sum())

# ---------- 6. FACT_SHIPMENT ----------
fact_shipment = base[[
    "shipment_id", "date_key", "carrier_key", "supplier_key", "route_key", "product_key",
    "order_quantity", "order_value", "total_logistics_cost", "distance_km",
    "delay_hours", "delay_days", "delivery_attempts",
    "delivery_delayed", "delivery_successful", "customs_required", "delay_reason",
    "transport_mode", "vehicle_type", "vehicle_age_bucket",
    "weather_condition", "traffic_congestion_level",
]]

# ---------- 7. FACT_COST ----------
cost_cols = ["fuel_cost", "transportation_cost", "warehouse_cost",
             "handling_cost", "customs_cost", "insurance_cost"]

fact_cost = base[["shipment_id", "date_key", "carrier_key", "supplier_key",
                  "route_key", "product_key"] + cost_cols].melt(
    id_vars=["shipment_id", "date_key", "carrier_key", "supplier_key",
             "route_key", "product_key"],
    value_vars=cost_cols,
    var_name="cost_type",
    value_name="cost_amount",
)

fact_cost["cost_type"] = fact_cost["cost_type"].map({
    "fuel_cost": "Fuel",
    "transportation_cost": "Transportation",
    "warehouse_cost": "Warehouse",
    "handling_cost": "Handling",
    "customs_cost": "Customs",
    "insurance_cost": "Insurance",
})

# ---------- 8. EXPORT + ZIP ----------
out_dir = "/kaggle/working/pbi_supply_chain"
os.makedirs(out_dir, exist_ok=True)

tables = {
    "Dim_Date": dim_date.drop(columns=["date"]),
    "Dim_Carrier": dim_carrier,
    "Dim_Supplier": dim_supplier,
    "Dim_Route": dim_route,
    "Dim_Product": dim_product,
    "Fact_Shipment": fact_shipment,
    "Fact_Cost": fact_cost,
}

for name, t in tables.items():
    t.to_csv(f"{out_dir}/{name}.csv", index=False)
    print(f"{name}: {t.shape[0]:,} rows × {t.shape[1]} cols")

# bucket sanity check
print("\nvehicle_age_bucket counts:")
print(df["vehicle_age_bucket"].value_counts().sort_index())

zip_path = "/kaggle/working/pbi_supply_chain_tables.zip"
with ZipFile(zip_path, "w") as z:
    for name in tables:
        z.write(f"{out_dir}/{name}.csv", arcname=f"{name}.csv")

print("\nZIP:", zip_path, "| Size (MB):", round(os.path.getsize(zip_path) / 1e6, 2))

date_key nulls: 0
carrier_key nulls: 0
supplier_key nulls: 0
route_key nulls: 0
product_key nulls: 0
Dim_Date: 1,641 rows × 8 cols
Dim_Carrier: 120 rows × 4 cols
Dim_Supplier: 550 rows × 5 cols
Dim_Route: 60 rows × 5 cols
Dim_Product: 160 rows × 4 cols
Fact_Shipment: 100,000 rows × 22 cols
Fact_Cost: 600,000 rows × 8 cols

vehicle_age_bucket counts:
vehicle_age_bucket
1-3 yrs      27170
10-11 yrs    18254
4-6 yrs      27166
7-9 yrs      27410
Name: count, dtype: int64

ZIP: /kaggle/working/pbi_supply_chain_tables.zip | Size (MB): 43.3


In [13]:
num_cols = df.select_dtypes("number").columns
corr = df[num_cols].corr()["delay_hours"].sort_values(ascending=False)
print(corr.head(15))
print("\n--- lowest ---")
print(corr.tail(10))

delay_hours                       1.000000
delay_days                        0.998123
delivery_delayed                  0.882215
delivery_attempts                 0.822204
actual_delivery_days              0.656648
supply_chain_risk_score           0.653314
traffic_delay_hours               0.397158
traffic_index                     0.397156
supplier_historical_delay_rate    0.214313
historical_delay_rate             0.202764
warehouse_utilization             0.159578
stockout_risk                     0.133593
backorder_quantity                0.097158
warehouse_capacity                0.027938
order_backlog                     0.010989
Name: delay_hours, dtype: float64

--- lowest ---
fuel_efficiency              -0.005846
vehicle_age                  -0.006319
supplier_lead_time_days      -0.007016
staffing_level               -0.011383
carrier_rating               -0.190573
carrier_reliability_score    -0.202744
on_time_delivery_rate        -0.202764
supplier_rating              -0.

In [14]:
# Is delivery_performance_score derived from delay?
print(df[["delay_hours", "delivery_performance_score"]].corr())

# Is supply_chain_risk_score derived from delay?
print(df[["delay_hours", "supply_chain_risk_score"]].corr())

                            delay_hours  delivery_performance_score
delay_hours                    1.000000                   -0.844383
delivery_performance_score    -0.844383                    1.000000
                         delay_hours  supply_chain_risk_score
delay_hours                 1.000000                 0.653314
supply_chain_risk_score     0.653314                 1.000000
